# Day 4 — Content-Based Filtering

**Goal:** Find movies similar to a movie the user already likes — using the movie's own metadata.

**Pipeline:**
```
Raw TMDB data
     ↓ ast.literal_eval
Parse JSON-like strings into real Python lists
     ↓ extract_names()
Build one text profile per movie
     ↓ TfidfVectorizer
Convert text to number vectors (10,000 dimensions)
     ↓ cosine_similarity
Measure angle between movie vectors
     ↓ argsort
Return top-N closest movies
```

**No users involved.** This model only looks at movie content.

In [ ]:
import pandas as pd
import numpy as np
import ast
import joblib
import scipy.sparse as sparse
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches

os.makedirs('../models', exist_ok=True)

print('Libraries loaded.')

## 1. Load TMDB metadata

Download from: https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata
File needed: `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv`

In [ ]:
tmdb_movies  = pd.read_csv('../data/tmdb_5000_movies.csv')
tmdb_credits = pd.read_csv('../data/tmdb_5000_credits.csv')

print(f'TMDB movies:  {tmdb_movies.shape}')
print(f'TMDB credits: {tmdb_credits.shape}')

# Merge credits into movies
tmdb_credits = tmdb_credits.rename(columns={'movie_id': 'id'})
tmdb_df = tmdb_movies.merge(tmdb_credits[['id', 'cast', 'crew']], on='id')
print(f'Merged: {tmdb_df.shape}')

## 2. Parse the JSON-like columns

Columns like `genres`, `cast`, `keywords` contain strings that LOOK like Python lists:
```
"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
```

Python can't loop over strings. `ast.literal_eval` converts them to real Python objects.

In [ ]:
# See the problem clearly
raw_val = tmdb_df['genres'].iloc[0]
print(f'Type before: {type(raw_val)}')  # str
print(f'Value: {raw_val[:80]}...')

parsed_val = ast.literal_eval(raw_val)
print(f'\nType after:  {type(parsed_val)}')  # list
print(f'First item:  {parsed_val[0]}')

In [ ]:
def extract_names(val, key='name', top_n=3):
    """
    Convert a stringified list of dicts to a space-separated string.
    
    Input:  "[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
    Output: "Action Adventure"
    
    top_n: only take the first top_n items (e.g. top 3 cast members)
    try/except: if any row is malformed, return empty string (don't crash)
    """
    try:
        items = ast.literal_eval(val)
        names = [x[key] for x in items[:top_n]]
        return ' '.join(names)
    except:
        return ''


def extract_director(crew_val):
    """
    Find the director from the crew list.
    crew is a list of dicts — we look for the one where job == 'Director'.
    """
    try:
        crew = ast.literal_eval(crew_val)
        directors = [p['name'] for p in crew if p.get('job') == 'Director']
        return ' '.join(directors)
    except:
        return ''


# Apply to every row
tmdb_df['genres_parsed']   = tmdb_df['genres'].apply(extract_names)
tmdb_df['keywords_parsed'] = tmdb_df['keywords'].apply(extract_names)
tmdb_df['cast_parsed']     = tmdb_df['cast'].apply(lambda x: extract_names(x, top_n=3))
tmdb_df['director_parsed'] = tmdb_df['crew'].apply(extract_director)

# Preview
tmdb_df[['title', 'genres_parsed', 'cast_parsed', 'director_parsed']].head(3)

## 3. Build movie profile

Combine all signals into one text string per movie.

Why write director twice? TF-IDF counts word frequency.
More occurrences = more weight. Director is the strongest style signal,
so we double it to give it more influence on similarity.

In [ ]:
tmdb_df['profile'] = (
    tmdb_df['genres_parsed']   + ' ' +
    tmdb_df['keywords_parsed'] + ' ' +
    tmdb_df['cast_parsed']     + ' ' +
    tmdb_df['director_parsed'] + ' ' +
    tmdb_df['director_parsed']       # director doubled for extra weight
)

# Check what a profile looks like
inception_row = tmdb_df[tmdb_df['title'] == 'Inception']
if not inception_row.empty:
    print('Inception profile:')
    print(inception_row['profile'].values[0])
else:
    print('Sample profile:')
    print(tmdb_df['profile'].iloc[0])

## 4. TF-IDF vectorisation

Converts text profiles into number vectors.

Each movie becomes a vector of 10,000 numbers.
Most are 0 (the movie doesn't use that word).
Non-zero values represent distinctive words — weighted by how rare they are.

Result: a sparse matrix of shape (n_movies, 10000)

In [ ]:
tfidf = TfidfVectorizer(
    stop_words='english',   # remove 'the', 'a', 'is' — no signal
    max_features=10000      # keep top 10,000 most informative words
)

# fit_transform: learn vocabulary + convert all profiles to vectors
tfidf_matrix = tfidf.fit_transform(tmdb_df['profile'].fillna(''))

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  rows    = {tfidf_matrix.shape[0]} movies')
print(f'  columns = {tfidf_matrix.shape[1]} unique words')
print(f'  type    = {type(tfidf_matrix)} (sparse, memory efficient)')

## 5. Build title → index lookup

MovieLens titles: "Toy Story (1995)"
TMDB titles:      "Toy Story"

We strip the year so they can match.

In [ ]:
movies_df = pd.read_csv('../data/movies.csv')

# Strip year from MovieLens titles: "Toy Story (1995)" → "Toy Story"
movies_df['title_clean'] = (
    movies_df['title']
    .str.replace(r'\s*\(\d{4}\)', '', regex=True)
    .str.strip()
)

# TMDB titles just need whitespace stripped
tmdb_df['title_clean'] = tmdb_df['title'].str.strip()

# dict: "Inception" → row 4512 in tfidf_matrix
title_to_idx = (
    pd.Series(tmdb_df.index, index=tmdb_df['title_clean'])
    .drop_duplicates()
)

print(f'Title index built. {len(title_to_idx):,} movies indexed.')

## 6. Content-based recommender function

In [ ]:
def recommend_content(movie_title, n=10):
    """
    Returns n most content-similar movies to the given title.
    
    Steps:
    1. Find the movie in our index (fuzzy match for typos)
    2. Get its TF-IDF vector
    3. Compute cosine similarity against all other movies
    4. Sort and return top-n
    """
    # Step 1: find the movie, handle typos
    if movie_title not in title_to_idx:
        close = get_close_matches(movie_title, title_to_idx.index, n=1, cutoff=0.6)
        if not close:
            return f"Could not find '{movie_title}'. Check the title spelling."
        movie_title = close[0]
        print(f"Did you mean: '{movie_title}'?")

    # Step 2: get this movie's TF-IDF vector (one row from the matrix)
    idx       = title_to_idx[movie_title]
    movie_vec = tfidf_matrix[idx]           # shape: (1, 10000)

    # Step 3: cosine similarity against ALL movies
    # Result: array of shape (1, n_movies)
    # Each value = similarity score between movie_vec and that movie
    sim_scores = cosine_similarity(movie_vec, tfidf_matrix).flatten()

    # Step 4: sort descending, skip index 0 (the movie itself, score=1.0)
    # argsort() → ascending indices, [::-1] → flip to descending
    # [1:n+1]   → skip self (index 0), take next n
    top_indices = np.argsort(sim_scores)[::-1][1:n + 1]

    result = tmdb_df.iloc[top_indices][['title', 'genres_parsed']].copy()
    result['similarity'] = sim_scores[top_indices].round(3)
    result = result.reset_index(drop=True)

    return result


# Test
print('Similar to The Dark Knight:')
print(recommend_content('The Dark Knight', n=10))

In [ ]:
# Test with a typo — fuzzy matching kicks in
print('Similar to "Inceptoin" (typo):')
print(recommend_content('Inceptoin', n=5))

In [ ]:
print('Similar to Toy Story:')
print(recommend_content('Toy Story', n=10))

## 7. Save everything

In [ ]:
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')
sparse.save_npz('../models/tfidf_matrix.npz', tfidf_matrix)

# Save TMDB profiles (needed for Day 5 content candidate generation)
tmdb_df[['title_clean', 'profile', 'genres_parsed', 'keywords_parsed',
          'cast_parsed', 'director_parsed']].to_parquet('../models/tmdb_profiles.parquet')

# Save the title→index mapping
title_to_idx.to_pickle('../models/title_to_idx.pkl')

# Save movies_df with clean title column
movies_df.to_parquet('../data/movies_clean.parquet', index=False)

print('Saved:')
print('  ../models/tfidf_vectorizer.pkl')
print('  ../models/tfidf_matrix.npz')
print('  ../models/tmdb_profiles.parquet')
print('  ../models/title_to_idx.pkl')
print('  ../data/movies_clean.parquet')